In [ ]:
      
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
气象数据温度预测 - GRU模型
使用除温度外的所有特征预测温度，序列长度50
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# 设置随机种子
np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用设备: {device}\n')

# ===================== 1. 数据加载与预处理 =====================
print('=' * 60)
print('1. 数据加载与预处理')
print('=' * 60)

# 加载数据
try:
    df = pd.read_csv('mpi_roof.csv', encoding='utf-8')
except:
    df = pd.read_csv('mpi_roof.csv', encoding='latin-1')

print(f'原始数据: {df.shape}')
print(f'时间范围: {df["Date Time"].iloc[0]} 至 {df["Date Time"].iloc[-1]}')

# 处理CO2异常值
df['CO2 (ppm)'] = df['CO2 (ppm)'].replace(-9999.0, np.nan)
df['CO2 (ppm)'].fillna(df['CO2 (ppm)'].median(), inplace=True)
print(f'\nCO2异常值已处理（中位数填充）')
print(f'CO2范围: {df["CO2 (ppm)"].min():.1f} ~ {df["CO2 (ppm)"].max():.1f} ppm')

# 移除高相关冗余特征
drop_cols = ['Date Time', 'T (degC)', 'Tpot (K)', 'Tlog (degC)', 
             'H2OC (mmol/mol)', 'sh (g/kg)']
feature_cols = [col for col in df.columns if col not in drop_cols]
target_col = 'T (degC)'

print(f'\n特征数量: {len(feature_cols)}')

# 提取特征和目标
X = df[feature_cols].values
y = df[target_col].values
print(f'X shape: {X.shape}, y shape: {y.shape}')

# ===================== 2. 数据标准化与划分 =====================
print('\n' + '=' * 60)
print('2. 数据标准化与划分')
print('=' * 60)

# 按时间顺序划分：前80%训练，后20%验证
split_idx = int(len(X) * 0.8)
X_train, X_val = X[:split_idx], X[split_idx:]
y_train, y_val = y[:split_idx], y[split_idx:]

print(f'训练集: {X_train.shape}, 验证集: {X_val.shape}')
print(f'划分点: 第 {split_idx} 条记录')

# 标准化（仅在训练集上fit）
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
X_val_scaled = scaler_X.transform(X_val)

y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
y_val_scaled = scaler_y.transform(y_val.reshape(-1, 1)).flatten()

print('数据标准化完成')

# ===================== 3. 构建时序数据集 =====================
print('\n' + '=' * 60)
print('3. 构建时序数据集')
print('=' * 60)

class TimeSeriesDataset(Dataset):
    """时序数据集"""
    def __init__(self, X, y, seq_len=50):
        self.X = X
        self.y = y
        self.seq_len = seq_len
    
    def __len__(self):
        return len(self.X) - self.seq_len
    
    def __getitem__(self, idx):
        X_seq = torch.FloatTensor(self.X[idx:idx+self.seq_len])
        y_target = torch.FloatTensor([self.y[idx+self.seq_len]])
        return X_seq, y_target

seq_len = 50
batch_size = 128

train_dataset = TimeSeriesDataset(X_train_scaled, y_train_scaled, seq_len)
val_dataset = TimeSeriesDataset(X_val_scaled, y_val_scaled, seq_len)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f'序列长度: {seq_len}')
print(f'批次大小: {batch_size}')
print(f'训练样本数: {len(train_dataset)}')
print(f'验证样本数: {len(val_dataset)}')

# ===================== 4. 定义GRU模型 =====================
print('\n' + '=' * 60)
print('4. 定义GRU模型')
print('=' * 60)

class GRUModel(nn.Module):
    """GRU温度预测模型"""
    def __init__(self, input_dim, hidden_dim=128, num_layers=2, dropout=0.2):
        super(GRUModel, self).__init__()
        
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        self.gru = nn.GRU(
            input_dim, 
            hidden_dim, 
            num_layers, 
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        self.fc = nn.Linear(hidden_dim, 1)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        gru_out, _ = self.gru(x)
        last_out = gru_out[:, -1, :]
        out = self.dropout(last_out)
        out = self.fc(out)
        return out

input_dim = X_train_scaled.shape[1]
model = GRUModel(input_dim=input_dim, hidden_dim=128, num_layers=2, dropout=0.2)
model = model.to(device)

print(model)
print(f'\n模型参数量: {sum(p.numel() for p in model.parameters()):,}')

# ===================== 5. 训练设置 =====================
print('\n' + '=' * 60)
print('5. 训练模型')
print('=' * 60)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

num_epochs = 10
train_losses_step = []  # 每个step的训练loss
val_losses_step = []    # 每个epoch后的验证loss（复制到所有step）
train_steps = []        # 训练step计数
val_steps = []          # 验证step计数
best_val_loss = float('inf')
global_step = 0

# ===================== 6. 训练循环 =====================
for epoch in range(num_epochs):
    # 训练
    model.train()
    epoch_train_loss = 0
    
    for batch_idx, (X_batch, y_batch) in enumerate(train_loader):
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        output = model(X_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        
        # 记录每个step的loss
        train_losses_step.append(loss.item())
        train_steps.append(global_step)
        epoch_train_loss += loss.item()
        global_step += 1
    
    avg_train_loss = epoch_train_loss / len(train_loader)
    
    # 验证
    model.eval()
    epoch_val_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            output = model(X_batch)
            loss = criterion(output, y_batch)
            epoch_val_loss += loss.item()
    
    avg_val_loss = epoch_val_loss / len(val_loader)
    
    # 记录验证loss（对应当前的global_step）
    val_losses_step.append(avg_val_loss)
    val_steps.append(global_step - 1)
    
    # 学习率调整
    scheduler.step(avg_val_loss)
    
    # 保存最佳模型
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), 'best_gru_model.pth')
        best_epoch = epoch + 1
    
    # 每个epoch都打印
    print(f'Epoch [{epoch+1:2d}/{num_epochs}] '
          f'Train Loss: {avg_train_loss:.4f}, '
          f'Val Loss: {avg_val_loss:.4f}, '
          f'Step: {global_step}')

print(f'\n训练完成！最佳验证损失: {best_val_loss:.4f} (Epoch {best_epoch})')

# ===================== 7. 模型评估 =====================
print('\n' + '=' * 60)
print('6. 模型评估')
print('=' * 60)

# 加载最佳模型
model.load_state_dict(torch.load('best_gru_model.pth'))
model.eval()

# 在验证集上预测
predictions = []
actuals = []

with torch.no_grad():
    for X_batch, y_batch in val_loader:
        X_batch = X_batch.to(device)
        output = model(X_batch)
        predictions.extend(output.cpu().numpy())
        actuals.extend(y_batch.numpy())

predictions = np.array(predictions).flatten()
actuals = np.array(actuals).flatten()

# 反标准化
predictions_real = scaler_y.inverse_transform(predictions.reshape(-1, 1)).flatten()
actuals_real = scaler_y.inverse_transform(actuals.reshape(-1, 1)).flatten()

# 计算评估指标
mae = np.mean(np.abs(predictions_real - actuals_real))
rmse = np.sqrt(np.mean((predictions_real - actuals_real)**2))
mape = np.mean(np.abs((actuals_real - predictions_real) / actuals_real)) * 100

print(f'验证集评估指标:')
print(f'  MAE:  {mae:.3f} °C')
print(f'  RMSE: {rmse:.3f} °C')
print(f'  MAPE: {mape:.2f} %')

# ===================== 8. 可视化 =====================
print('\n' + '=' * 60)
print('7. 生成可视化图表')
print('=' * 60)

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 图1: 训练曲线（以step为单位）
plt.figure(figsize=(12, 5))
plt.plot(train_steps, train_losses_step, label='Train Loss', linewidth=1, alpha=0.7)
plt.plot(val_steps, val_losses_step, label='Val Loss', linewidth=2, marker='o', markersize=4)
plt.xlabel('Training Step', fontsize=12)
plt.ylabel('Loss (MSE)', fontsize=12)
plt.title('训练和验证损失曲线（按Step）', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('training_curve.png', dpi=300)
print('✓ 已保存: training_curve.png')
plt.close()

# 图2: 预测 vs 实际
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# 时间序列对比（前1000个点）
plot_len = min(1000, len(predictions_real))
axes[0].plot(actuals_real[:plot_len], label='实际温度', linewidth=1.5, alpha=0.8)
axes[0].plot(predictions_real[:plot_len], label='预测温度', linewidth=1.5, alpha=0.8)
axes[0].set_xlabel('时间步', fontsize=11)
axes[0].set_ylabel('温度 (°C)', fontsize=11)
axes[0].set_title('预测 vs 实际温度（前1000个点）', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# 散点图
axes[1].scatter(actuals_real, predictions_real, alpha=0.3, s=10)
min_val = min(actuals_real.min(), predictions_real.min())
max_val = max(actuals_real.max(), predictions_real.max())
axes[1].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='理想预测线')
axes[1].set_xlabel('实际温度 (°C)', fontsize=11)
axes[1].set_ylabel('预测温度 (°C)', fontsize=11)
axes[1].set_title('预测准确性散点图', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('prediction_results.png', dpi=300)
print('✓ 已保存: prediction_results.png')
plt.close()

# 图3: 误差分析
errors = predictions_real - actuals_real

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.hist(errors, bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('预测误差 (°C)', fontsize=11)
plt.ylabel('频数', fontsize=11)
plt.title('预测误差分布', fontsize=13, fontweight='bold')
plt.axvline(x=0, color='r', linestyle='--', linewidth=2)
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.boxplot(errors, vert=True)
plt.ylabel('预测误差 (°C)', fontsize=11)
plt.title('误差箱线图', fontsize=13, fontweight='bold')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('error_analysis.png', dpi=300)
print('✓ 已保存: error_analysis.png')
plt.close()

print(f'\n误差统计:')
print(f'  均值: {errors.mean():.3f} °C')
print(f'  标准差: {errors.std():.3f} °C')
print(f'  中位数: {np.median(errors):.3f} °C')

# ===================== 9. 总结 =====================
print('\n' + '=' * 60)
print('8. 总结')
print('=' * 60)
print(f'\n模型配置:')
print(f'  - 输入特征: {input_dim}个气象参数')
print(f'  - 序列长度: {seq_len}个时间步')
print(f'  - GRU隐藏层: 128维，2层')
print(f'  - Dropout: 0.2')
print(f'  - 训练轮数: {num_epochs}')
print(f'  - 总训练步数: {global_step}')
print(f'\n性能指标:')
print(f'  - MAE:  {mae:.3f} °C')
print(f'  - RMSE: {rmse:.3f} °C')
print(f'  - MAPE: {mape:.2f} %')
print(f'  - 最佳验证损失: {best_val_loss:.4f}')
print(f'\n生成的文件:')
print(f'  ✓ best_gru_model.pth (最佳模型)')
print(f'  ✓ training_curve.png')
print(f'  ✓ prediction_results.png')
print(f'  ✓ error_analysis.png')

print('\n' + '=' * 60)
print('完成！')
print('=' * 60)

    